In [3]:
"""
Сравнение структур Chai-1 с pathA, pathB и original.

Метрики:
  - Global RMSD   : Chai vs pathA, pathB, original (по всем CA)
  - Scaffold RMSD : только по остаткам из gap_resseq (metadata.tsv)
  - Scaffold seqid: sequence identity в scaffold регионе

Запуск:
    cd /home/domain/aristowi/la-proteina-main
    python analyze_motif_scaffolding.py
"""

import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

try:
    from Bio.PDB import PDBParser, MMCIFParser, Superimposer
    import Bio.PDB as PDB
except ImportError:
    raise ImportError("pip install biopython")

# ─── Пути ────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path("/home/domain/aristowi/la-proteina-main")
CHAI_OUTPUT    = PROJECT_ROOT / "chai_output"
INFERENCE_ROOT = PROJECT_ROOT / "inference"
DATASET_DIR    = PROJECT_ROOT / "inpainting_dataset"
METADATA_TSV   = DATASET_DIR / "metadata.tsv"
OUT_CSV        = PROJECT_ROOT / "motif_scaffolding_analysis.csv"
OUT_SUMMARY    = PROJECT_ROOT / "motif_scaffolding_summary.txt"

# ─── Вспомогательные функции ─────────────────────────────────────────────────

def get_ca_atoms(path: Path, is_cif: bool = False):
    """Возвращает dict {resseq: CA_atom} из первой модели/цепи."""
    try:
        parser = MMCIFParser(QUIET=True) if is_cif else PDBParser(QUIET=True)
        structure = parser.get_structure("p", str(path))
        ca_dict = {}
        for model in structure:
            for chain in model:
                for residue in chain:
                    if PDB.is_aa(residue, standard=True) and "CA" in residue:
                        resseq = residue.get_id()[1]
                        ca_dict[resseq] = residue["CA"]
            break
        return ca_dict
    except Exception as e:
        print(f"  [WARN] не удалось прочитать {path.name}: {e}")
        return {}


def calc_rmsd_from_atoms(atoms_ref, atoms_mob):
    """RMSD после суперпозиции Kabsch. Принимает списки атомов."""
    if len(atoms_ref) < 3 or len(atoms_mob) < 3:
        return None
    n = min(len(atoms_ref), len(atoms_mob))
    try:
        sup = Superimposer()
        sup.set_atoms(atoms_ref[:n], atoms_mob[:n])
        sup.apply(atoms_mob[:n])
        return round(float(sup.rms), 3)
    except Exception as e:
        return None


def calc_rmsd_subset(ca_ref: dict, ca_mob: dict, resseqs: list):
    """
    RMSD по подмножеству resseq.
    Берём только те позиции которые есть в обоих структурах.
    """
    common = [r for r in resseqs if r in ca_ref and r in ca_mob]
    if len(common) < 3:
        return None, len(common)
    atoms_ref = [ca_ref[r] for r in common]
    atoms_mob = [ca_mob[r] for r in common]
    rmsd = calc_rmsd_from_atoms(atoms_ref, atoms_mob)
    return rmsd, len(common)


def calc_rmsd_global(ca_ref: dict, ca_mob: dict):
    """RMSD по всем общим позициям."""
    common = sorted(set(ca_ref.keys()) & set(ca_mob.keys()))
    if len(common) < 3:
        return None, len(common)
    atoms_ref = [ca_ref[r] for r in common]
    atoms_mob = [ca_mob[r] for r in common]
    return calc_rmsd_from_atoms(atoms_ref, atoms_mob), len(common)


def seq_identity_subset(ca_ref: dict, ca_mob: dict, resseqs: list):
    """
    Sequence identity в scaffold регионе.
    Сравниваем названия остатков по общим позициям.
    """
    common = [r for r in resseqs if r in ca_ref and r in ca_mob]
    if not common:
        return None, 0
    matches = sum(
        ca_ref[r].get_parent().get_resname() ==
        ca_mob[r].get_parent().get_resname()
        for r in common
    )
    return round(matches / len(common), 3), len(common)


def extract_sample_id(chai_folder_name: str):
    """
    'job_0_id_0_motif_102m_A_seq00' → '102m_A'
    Паттерн: job_\d+_id_\d+_motif_<sample_id>_seq\d+
    """
    m = re.match(r"^job_\d+_id_\d+_motif_(.+)_seq\d+$", chai_folder_name)
    return m.group(1) if m else None


def find_laproteina_paths(sample_id: str):
    """
    Ищет pathA и pathB для данного sample_id.
    Путь: inference/inpainting_{sid}_{sid}/job_0_id_0_motif_{sid}/
    """
    # Паттерн из наблюдений: inpainting_102m_A_102m_A
    infer_dir = INFERENCE_ROOT / f"inpainting_{sample_id}_{sample_id}"
    sub_dir   = infer_dir / f"job_0_id_0_motif_{sample_id}"

    path_a = sub_dir / f"job_0_id_0_motif_{sample_id}_pathA.pdb"
    path_b = sub_dir / f"job_0_id_0_motif_{sample_id}_pathB.pdb"

    # Fallback: поиск по glob если имя немного другое
    if not path_a.exists():
        found = list(infer_dir.glob(f"**/*pathA.pdb"))
        path_a = found[0] if found else path_a
    if not path_b.exists():
        found = list(infer_dir.glob(f"**/*pathB.pdb"))
        path_b = found[0] if found else path_b

    return (path_a if path_a.exists() else None,
            path_b if path_b.exists() else None)


def find_original(sample_id: str):
    """original.pdb из inpainting_dataset."""
    p = DATASET_DIR / sample_id / "original.pdb"
    return p if p.exists() else None


# ─── Загрузка metadata ───────────────────────────────────────────────────────

def load_metadata():
    """
    Загружает metadata.tsv и возвращает dict:
      sample_id → list of scaffold resseq (int)
    """
    if not METADATA_TSV.exists():
        print(f"[WARN] metadata.tsv не найден: {METADATA_TSV}")
        return {}
    meta = pd.read_csv(METADATA_TSV, sep="\t")
    result = {}
    for _, row in meta.iterrows():
        sid = row["sample_id"]
        gap_resseq_str = str(row.get("gap_resseq", ""))
        if gap_resseq_str and gap_resseq_str != "nan":
            resseqs = [int(x.strip()) for x in gap_resseq_str.split(",")
                       if x.strip().lstrip("-").isdigit()]
        else:
            resseqs = []
        result[sid] = resseqs
    print(f"Загружена metadata для {len(result)} образцов")
    return result


# ─── Основной цикл ────────────────────────────────────────────────────────────

def main():
    metadata = load_metadata()

    # Собираем все Chai-папки для motif
    chai_dirs = sorted([
        d for d in CHAI_OUTPUT.iterdir()
        if d.is_dir()
        and not d.name.startswith(".")
        and "motif" in d.name
    ])

    if not chai_dirs:
        print(f"[ERROR] Нет папок с motif в {CHAI_OUTPUT}")
        return

    # Группируем по sample_id
    by_sample = defaultdict(list)
    for d in chai_dirs:
        sid = extract_sample_id(d.name)
        if sid:
            by_sample[sid].append(d)
        else:
            print(f"[SKIP] непонятное имя: {d.name}")

    print(f"Найдено {len(chai_dirs)} Chai-папок → {len(by_sample)} образцов\n")

    rows = []

    for sample_id in sorted(by_sample.keys()):
        path_a    = None
        path_b    = None
        path_orig = None

        path_a, path_b = find_laproteina_paths(sample_id)
        path_orig      = find_original(sample_id)

        scaffold_resseqs = metadata.get(sample_id, [])

        print(f"── {sample_id}  "
              f"[A={'OK' if path_a else 'MISS'}  "
              f"B={'OK' if path_b else 'MISS'}  "
              f"orig={'OK' if path_orig else 'MISS'}  "
              f"scaffold={len(scaffold_resseqs)}res]")

        # Читаем структуры La-Proteina и original один раз
        ca_A    = get_ca_atoms(path_a)    if path_a    else {}
        ca_B    = get_ca_atoms(path_b)    if path_b    else {}
        ca_orig = get_ca_atoms(path_orig) if path_orig else {}

        # RMSD между pathA и pathB (насколько разные конформации)
        rmsd_AB_global, n_AB = calc_rmsd_global(ca_A, ca_B)
        rmsd_AB_scaf,   _    = calc_rmsd_subset(ca_A, ca_B, scaffold_resseqs)

        for chai_dir in sorted(by_sample[sample_id]):
            seq_tag   = chai_dir.name
            cif_files = sorted(chai_dir.glob("pred.model_idx_*.cif"))
            score_files = {
                int(re.search(r"model_idx_(\d+)", f.name).group(1)): f
                for f in chai_dir.glob("scores.model_idx_*.npz")
                if re.search(r"model_idx_(\d+)", f.name)
            }

            # Читаем scores
            model_scores = {}
            for idx, sf in score_files.items():
                try:
                    import numpy as np
                    data = np.load(str(sf))
                    for key in ["aggregate_score", "ptm", "plddt"]:
                        if key in data:
                            model_scores[idx] = round(float(np.mean(data[key])), 4)
                            break
                except Exception:
                    pass

            for cif_file in cif_files:
                m = re.search(r"model_idx_(\d+)", cif_file.name)
                model_idx  = int(m.group(1)) if m else -1
                chai_score = model_scores.get(model_idx)

                ca_chai = get_ca_atoms(cif_file, is_cif=True)

                # ── Global RMSD ──────────────────────────────────────────
                rmsd_chai_A_glob,    n_glob_A = calc_rmsd_global(ca_A,    ca_chai)
                rmsd_chai_B_glob,    n_glob_B = calc_rmsd_global(ca_B,    ca_chai)
                rmsd_chai_orig_glob, n_glob_o = calc_rmsd_global(ca_orig, ca_chai)

                # ── Scaffold RMSD ────────────────────────────────────────
                rmsd_chai_A_scaf, n_scaf_A = calc_rmsd_subset(ca_A,    ca_chai, scaffold_resseqs)
                rmsd_chai_B_scaf, n_scaf_B = calc_rmsd_subset(ca_B,    ca_chai, scaffold_resseqs)
                rmsd_chai_orig_scaf, n_scaf_o = calc_rmsd_subset(ca_orig, ca_chai, scaffold_resseqs)

                # ── Scaffold Sequence Identity ───────────────────────────
                seqid_A_vs_B_scaf, _ = seq_identity_subset(ca_A, ca_B, scaffold_resseqs)
                seqid_chai_orig, _ = seq_identity_subset(ca_orig, ca_chai, scaffold_resseqs)

                # Chai ближе к A или B по scaffold RMSD
                if rmsd_chai_A_scaf is not None and rmsd_chai_B_scaf is not None:
                    closer_scaf = "A" if rmsd_chai_A_scaf <= rmsd_chai_B_scaf else "B"
                elif rmsd_chai_A_glob is not None and rmsd_chai_B_glob is not None:
                    closer_scaf = "A" if rmsd_chai_A_glob <= rmsd_chai_B_glob else "B"
                else:
                    closer_scaf = "?"

                row = {
                    "sample_id":            sample_id,
                    "seq_tag":              seq_tag,
                    "model_idx":            model_idx,
                    "chai_score":           chai_score,
                    "n_scaffold_res":       len(scaffold_resseqs),
                    # La-Proteina A vs B
                    "rmsd_A_vs_B_global":   rmsd_AB_global,
                    "rmsd_A_vs_B_scaffold": rmsd_AB_scaf,
                    # Global RMSD: Chai vs ...
                    "rmsd_chai_A_global":   rmsd_chai_A_glob,
                    "rmsd_chai_B_global":   rmsd_chai_B_glob,
                    "rmsd_chai_orig_global":rmsd_chai_orig_glob,
                    # Scaffold RMSD: Chai vs ...
                    "rmsd_chai_A_scaf":     rmsd_chai_A_scaf,
                    "rmsd_chai_B_scaf":     rmsd_chai_B_scaf,
                    "rmsd_chai_orig_scaf":  rmsd_chai_orig_scaf,
                    # Scaffold seq identity: Chai vs ...
                    "seqid_A_vs_B_scaf":    seqid_A_vs_B_scaf,
                    # Closer to
                    "closer_to_scaf":       closer_scaf,
                    # Atom counts
                    "n_common_glob_A":      n_glob_A,
                    "n_common_scaf_A":      n_scaf_A,
                }
                rows.append(row)

                print(f"   {seq_tag} m{model_idx}: "
                      f"score={chai_score}  "
                      f"glob→A={rmsd_chai_A_glob}Å B={rmsd_chai_B_glob}Å orig={rmsd_chai_orig_glob}Å  "
                      f"scaf→A={rmsd_chai_A_scaf}Å B={rmsd_chai_B_scaf}Å  "
                      f"seqid_scaf={seqid_chai_orig}  closer={closer_scaf}")
        print()

    if not rows:
        print("[ERROR] Нет данных")
        return

    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False)
    print(f"✓ Детали: {OUT_CSV}\n")

    # ── Лучшая модель по score для сводки ────────────────────────────────────
    if df["chai_score"].notna().any():
        df_best = (df.sort_values("chai_score", ascending=False)
                     .groupby(["sample_id", "seq_tag"], as_index=False)
                     .first())
    else:
        df_best = df[df["model_idx"] == 0].copy()

    # ── Сводная статистика ────────────────────────────────────────────────────
    lines = ["=" * 70, "СВОДНАЯ СТАТИСТИКА", "=" * 70]
    lines.append(f"\nОбразцов:       {df['sample_id'].nunique()}")
    lines.append(f"Chai seq_tags:  {df['seq_tag'].nunique()}")
    lines.append(f"Chai моделей:   {len(df)}")
    lines.append(f"(сводка по лучшей модели, n={len(df_best)})")

    def stat_line(series, label):
        s = series.dropna()
        if len(s) == 0:
            return f"  {label}: нет данных"
        return (f"  {label}: "
                f"mean={s.mean():.2f}  median={s.median():.2f}  "
                f"min={s.min():.2f}  max={s.max():.2f}")

    lines.append("\n── La-Proteina: разность конформаций A↔B ──")
    lines.append(stat_line(df_best["rmsd_A_vs_B_global"],   "RMSD global"))
    lines.append(stat_line(df_best["rmsd_A_vs_B_scaffold"], "RMSD scaffold"))

    lines.append("\n── Chai vs La-Proteina: GLOBAL RMSD ──")
    lines.append(stat_line(df_best["rmsd_chai_A_global"],    "→ pathA"))
    lines.append(stat_line(df_best["rmsd_chai_B_global"],    "→ pathB"))
    lines.append(stat_line(df_best["rmsd_chai_orig_global"], "→ original"))

    lines.append("\n── Chai vs La-Proteina: SCAFFOLD RMSD ──")
    lines.append(stat_line(df_best["rmsd_chai_A_scaf"],    "→ pathA"))
    lines.append(stat_line(df_best["rmsd_chai_B_scaf"],    "→ pathB"))
    lines.append(stat_line(df_best["rmsd_chai_orig_scaf"], "→ original"))

    lines.append("\n── Chai vs original: SCAFFOLD Seq Identity ──")
    lines.append(stat_line(df_best["seqid_chai_orig_scaf"], "seqid"))
    lines.append(stat_line(df_best["seqid_chai_A_scaf"],    "seqid vs pathA"))
    lines.append(stat_line(df_best["seqid_chai_B_scaf"],    "seqid vs pathB"))

    counts = df_best["closer_to_scaf"].value_counts()
    total  = len(df_best)
    lines.append(f"\n── Chai ближе к (scaffold RMSD) ──")
    lines.append(f"  pathA: {counts.get('A',0)}/{total} ({counts.get('A',0)/total*100:.1f}%)")
    lines.append(f"  pathB: {counts.get('B',0)}/{total} ({counts.get('B',0)/total*100:.1f}%)")

    # Топ-10 лучших по scaffold RMSD к original
    lines.append("\n── Топ-10 по scaffold RMSD к original ──")
    top = df_best.nsmallest(10, "rmsd_chai_orig_scaf")[
        ["sample_id","seq_tag","rmsd_chai_orig_scaf",
         "rmsd_chai_A_scaf","rmsd_chai_B_scaf",
         "rmsd_A_vs_B_scaffold","seqid_chai_orig_scaf","chai_score"]
    ]
    lines.append(top.to_string(index=False))

    # Интерпретация
    lines.append("\n── Интерпретация ──")
    m_orig = df_best["rmsd_chai_orig_scaf"].dropna().mean()
    m_ab   = df_best["rmsd_A_vs_B_scaffold"].dropna().mean()
    m_glob = df_best["rmsd_chai_orig_global"].dropna().mean()

    if m_orig < 2.0:
        lines.append(f"  ✓ Scaffold RMSD к original < 2Å ({m_orig:.2f}) — петля восстановлена отлично")
    elif m_orig < 4.0:
        lines.append(f"  ~ Scaffold RMSD к original {m_orig:.2f}Å — умеренное восстановление")
    else:
        lines.append(f"  ✗ Scaffold RMSD к original {m_orig:.2f}Å — петли сильно отличаются")

    if m_glob < 3.0:
        lines.append(f"  ✓ Global RMSD к original {m_glob:.2f}Å — глобальный фолд правильный")
    elif m_glob < 6.0:
        lines.append(f"  ~ Global RMSD к original {m_glob:.2f}Å — глобальный фолд примерно верный")
    else:
        lines.append(f"  ✗ Global RMSD к original {m_glob:.2f}Å — глобальный фолд нарушен")

    if m_ab > 3.0:
        lines.append(f"  ✓ A↔B scaffold RMSD {m_ab:.2f}Å — dual-path генерирует разные конформации петли")
    else:
        lines.append(f"  ~ A↔B scaffold RMSD {m_ab:.2f}Å — конформации петли недостаточно различаются")

    lines.append("=" * 70)

    summary = "\n".join(lines)
    print(summary)
    with open(OUT_SUMMARY, "w") as f:
        f.write(summary + "\n")
    print(f"\n✓ Сводка: {OUT_SUMMARY}")


if __name__ == "__main__":
    main()

Загружена metadata для 100 образцов
Найдено 168 Chai-папок → 85 образцов

── 102l_A  [A=OK  B=OK  orig=OK  scaffold=34res]
   job_0_id_0_motif_102l_A_seq00 m0: score=0.1761  glob→A=1.354Å B=1.755Å orig=3.407Å  scaf→A=0.54Å B=2.503Å  seqid_scaf=0.029  closer=A
   job_0_id_0_motif_102l_A_seq00 m1: score=0.1759  glob→A=1.075Å B=1.569Å orig=3.321Å  scaf→A=0.571Å B=2.558Å  seqid_scaf=0.029  closer=A
   job_0_id_0_motif_102l_A_seq01 m0: score=0.1788  glob→A=1.096Å B=1.549Å orig=3.33Å  scaf→A=0.528Å B=2.443Å  seqid_scaf=0.029  closer=A
   job_0_id_0_motif_102l_A_seq01 m1: score=0.1792  glob→A=0.888Å B=1.427Å orig=3.254Å  scaf→A=0.63Å B=2.509Å  seqid_scaf=0.029  closer=A

── 102m_A  [A=MISS  B=MISS  orig=MISS  scaffold=0res]

── 105m_A  [A=OK  B=OK  orig=OK  scaffold=44res]
   job_0_id_0_motif_105m_A_seq00 m0: score=0.1873  glob→A=0.949Å B=0.961Å orig=1.817Å  scaf→A=0.703Å B=0.739Å  seqid_scaf=0.159  closer=A
   job_0_id_0_motif_105m_A_seq00 m1: score=0.1872  glob→A=0.965Å B=0.977Å orig=1.84Å 

KeyError: 'seqid_chai_orig_scaf'

In [16]:
import pandas as pd

df = pd.read_csv('motif_scaffolding_analysis.csv')
best = df.sort_values(['rmsd_A_vs_B_scaffold', 'seqid_A_vs_B_scaf'], ascending=[False, False])
best[['sample_id', 'seq_tag', 'rmsd_A_vs_B_global', 'rmsd_A_vs_B_scaffold', 'seqid_A_vs_B_scaf']].dropna()

,sample_id,seq_tag,rmsd_A_vs_B_global,rmsd_A_vs_B_scaffold,seqid_A_vs_B_scaf
280,1atk_A,job_0_id_0_motif_1atk_A_seq00,6.002,11.135,0.921
281,1atk_A,job_0_id_0_motif_1atk_A_seq00,6.002,11.135,0.921
282,1atk_A,job_0_id_0_motif_1atk_A_seq01,6.002,11.135,0.921
283,1atk_A,job_0_id_0_motif_1atk_A_seq01,6.002,11.135,0.921
212,1akv_A,job_0_id_0_motif_1akv_A_seq00,7.634,9.867,0.897
...,...,...,...,...,...
279,1asw_A,job_0_id_0_motif_1asw_A_seq01,5.557,0.002,1.000
320,1az8_A,job_0_id_0_motif_1az8_A_seq00,2.395,0.002,1.000
321,1az8_A,job_0_id_0_motif_1az8_A_seq00,2.395,0.002,1.000
322,1az8_A,job_0_id_0_motif_1az8_A_seq01,2.395,0.002,1.000
